# Code to reproduce the statistics reported in Table 2 of the report

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

(forgive the horrible column naming, its from an Excel workbook)

In [2]:
T_RES_COLS = ['Res 1', 'Res 2', 'Res 3 ', 'Res 4', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10']
S_RES_COLS = ['Res 1', 'Res 2', 'Res 3', 'Res 4', 'Res 5', 'Res 6', 'Res 7', 'Res 8','Res 9', 'Res 10', 'Res 11', 'Res 12', 'Res 13', 'Res 14', 'Res 15', 'Res 16', 'Res 17', 'Res 18', 'Res 19', 'Res 20', 'Res 21']

transect = pd.read_csv('../../../datastore/Transect.csv')
ssdf = pd.read_csv('../../../datastore/SCDs.csv')

# transect data
t_data = transect[T_RES_COLS].copy()
t_data['SCD'] = False

# scd data
s_data = ssdf[S_RES_COLS].copy()
s_data['SCD'] = True

# total df
df = pd.concat([t_data, s_data], axis=0)

df['mean'] = df[df.select_dtypes(include=[np.number]).columns].mean(axis=1)
df['median'] = df[df.select_dtypes(include=[np.number]).columns].median(axis=1)
df['max'] = df[df.select_dtypes(include=[np.number]).columns].max(axis=1)

In [3]:
df

,Res 1,Res 2,Res 3,Res 4,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,SCD,Res 3,...,Res 15,Res 16,Res 17,Res 18,Res 19,Res 20,Res 21,mean,median,max
0,3,3,3.0,3,NaN,NaN,NaN,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.000000,3.000000,3.0
1,5,46,41.0,22,NaN,NaN,NaN,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.500000,28.500000,46.0
2,2,2,1.0,40,NaN,NaN,NaN,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.250000,2.000000,40.0
3,4,5,13.0,9,NaN,NaN,NaN,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.750000,7.750000,13.0
4,1,3,1.0,2,NaN,NaN,NaN,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.750000,1.750000,3.0
5,3,3,3.0,0,NaN,NaN,NaN,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.250000,3.000000,3.0
6,1,3,2.0,13,NaN,NaN,NaN,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.750000,3.000000,13.0
7,3,3,2.0,7,NaN,NaN,NaN,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.750000,3.000000,7.0
8,4,5,2.0,1,NaN,NaN,NaN,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.000000,3.000000,5.0
9,0,2,1.0,6,NaN,NaN,NaN,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.250000,2.000000,6.0


In [4]:
def get_mad(df):
    return (df['mean'] - df['mean'].median()).abs().median()

def background(df):
    mad = get_mad(df)
    median = df['mean'].median()
    background_level = median + 2 * mad
    return background_level

In [5]:
def get_concentration_stats(df):
    transect_mean = df[df['SCD'] == False]['mean'].mean()
    scd_mean = df[df['SCD'] == True]['mean'].mean()

    transect_median = df[df['SCD'] == False]['mean'].median()
    scd_median = df[df['SCD'] == True]['mean'].median()

    transect_mad = get_mad(df[df['SCD'] == False])
    scd_mad = get_mad(df[df['SCD'] == True])

    return {
        'transect_mean': transect_mean,
        'scd_mean': scd_mean,
        'transect_median': transect_median,
        'scd_median': scd_median,
        'transect_mad': transect_mad,
        'scd_mad': scd_mad
    }

In [6]:
print(f'Dataset Stats: \nSCD Mean: {get_concentration_stats(df)["scd_mean"]:.3f} \
    \nTransect Mean: {get_concentration_stats(df)["transect_mean"]:.3f} \nSCD Median: {get_concentration_stats(df)["scd_median"]:.3f} \n\
Transect Median: {get_concentration_stats(df)["transect_median"]:.3f} \nSCD MAD: {get_concentration_stats(df)["scd_mad"]:.3f} \n\
Transect MAD: {get_concentration_stats(df)["transect_mad"]:.3f}')

print(f'MAD: {get_mad(df):.3f}')
print(f'Background Level: {background(df):.3f}')

Dataset Stats: 
SCD Mean: 12.037     
Transect Mean: 11.637 
SCD Median: 9.308 
Transect Median: 4.750 
SCD MAD: 5.708 
Transect MAD: 2.000
MAD: 3.200
Background Level: 12.900


Number of points above background level

In [7]:
print(f'number of points above background: {len(df[df["mean"] > background(df)])}')

number of points above background: 8


Concentrations above 40 ppm

In [8]:
print(f'Number of sites with maximum concentration above 40 ppm: {len(df[df["max"] >= 40])}')

Number of sites with maximum concentration above 40 ppm: 9


In [9]:
print(f'SCDs above background: {len(df[(df["SCD"] == True) & (df["mean"] > background(df))])}')

SCDs above background: 4


In [10]:
print(f'Means of maximum concentrations at SCD sites: {df[df["SCD"] == True]["max"].mean():.3f}')
print(f'Means of maximum concentrations at non-SCD sites: {df[df["SCD"] == False]["max"].mean():.3f}')

Means of maximum concentrations at SCD sites: 30.267
Means of maximum concentrations at non-SCD sites: 21.765


Mann Whitney population analysis

In [16]:
SCDs = df[df['SCD'] == True]
non_SCDs = df[df['SCD'] == False]

def get_mannwhitneyu(SCDs, non_SCDs, field='mean'):
    u_stat, u_p = stats.mannwhitneyu(SCDs[S_RES_COLS].stack(), non_SCDs[T_RES_COLS].stack(), alternative='two-sided')
    return u_stat, u_p

print(f'Mann-Whitney on maximum concentrations: U-statistic={get_mannwhitneyu(SCDs, non_SCDs, field="max")[0]:.3f}, p-value={get_mannwhitneyu(SCDs, non_SCDs, field="max")[1]:.3f}')
print(f'Mann-Whitney on mean concentrations: U-statistic={get_mannwhitneyu(SCDs, non_SCDs, field="mean")[0]:.3f}, p-value={get_mannwhitneyu(SCDs, non_SCDs, field="mean")[1]:.3f}')

Mann-Whitney on maximum concentrations: U-statistic=nan, p-value=nan
Mann-Whitney on mean concentrations: U-statistic=nan, p-value=nan


In [12]:
print(f'Median of peak concentrations at SCDs: {SCDs["max"].mean():.3f}')

Median of peak concentrations at SCDs: 30.267


In [13]:
print(f'Stats of SCD population: count={len(SCDs)}, max={SCDs["max"].max():.3f}, mean={SCDs["mean"].mean():.3f}, median={SCDs["mean"].median():.3f}, std={SCDs["mean"].std():.3f}')
print(f'Stats of non-SCD population: count={len(non_SCDs)}, max={non_SCDs["max"].max():.3f}, mean={non_SCDs["mean"].mean():.3f}, median={non_SCDs["mean"].median():.3f}, std={non_SCDs["mean"].std():.3f}')
print(f'stats of whole population: count={len(df)}, max={df["max"].max():.3f}, mean={df["mean"].mean():.3f}, median={df["mean"].median():.3f}, std={df["mean"].std():.3f}')

Stats of SCD population: count=15, max=106.000, mean=12.037, median=9.308, std=11.028
Stats of non-SCD population: count=34, max=300.000, mean=11.637, median=4.750, std=26.732
stats of whole population: count=49, max=300.000, mean=11.759, median=6.500, std=22.952
